In [1]:
library(tidyverse)
library(vegan)
options(stringsAsFactors = FALSE)

# =============================================================================
# 1. READ ALL DATASETS
# =============================================================================


# AMO Data
amo_table <- read.table("AMO/amo_monthly.txt", header = FALSE, skip = 1)
nm2 <- setNames(sprintf("%02d", 1:12), paste0("V", 2:13))

amo_ds <- amo_table %>%
  gather(Month, AMO, -V1) %>%
  mutate(
    Month = nm2[Month],
    time_month = paste(Month, V1, sep = "-")
  ) %>%
  select(time_month, AMO)

# MEI v.2 Data
meiv2_lines <- readLines("MEIv2/meiv2.data")
meiv2_lines <- meiv2_lines[c(-1, -48:-51)]

meiv2_ds <- read.table(textConnection(meiv2_lines), header = FALSE) %>%
  gather(Month, MEIv2, -V1) %>%
  mutate(
    Month = nm2[Month],
    time_month = paste(Month, V1, sep = "-")
  ) %>%
  select(time_month, MEIv2)


# ERA5 Data
wind_ds <- readRDS("processed/ERA5_data.rds")
wind_ds$date <- as.Date(wind_ds$time)
wind_ds$time_month <- format(wind_ds$date, format = "%m-%Y")
wind_ds <- wind_ds %>% select(-date, -time)

# Niskin Data (NOTE: Chl, PP, nutrients are depth-integrated)
niskin_ds <- readRDS("processed/Niskin_qchecked_100m.rds")
niskin_ds$time_month <- format(niskin_ds$date, format = "%m-%Y")
niskin_ds <- niskin_ds %>% select(-date)

# Apply Primary Productivity unit conversion (from Script 2)
niskin_ds$PrimaryProductivity <- niskin_ds$PrimaryProductivity * 12

# HPLC Data (NOTE: pigments are depth-integrated, size fractions are relative)
hplc_ds <- readRDS("processed/HPLC_interpolated_sizes.rds") %>%
  mutate(time_month = format(date, format = "%m-%Y")) %>%
  select(-date, -source, -DP2)

# Pigment diversity (Shannon, Pielou, Richness across pigment types)
pigment_cols <- c("Lut", "Pras", "Fuco", "Perid", "Allo", "But_fuco",
                  "Hex_fuco", "Zea", "Tot_Chl_b", "Tot_Chl_a", "Chl_c1c2", "Chl_c3")

pigment_ds <- hplc_ds %>%
  select(time_month, all_of(pigment_cols)) %>%
  filter(rowSums(is.na(pick(all_of(pigment_cols)))) < 6) %>%
  mutate(across(all_of(pigment_cols), ~replace_na(.x, 0)))

pigment_diversity <- pigment_ds %>%
  mutate(
    Shannon_pigment = vegan::diversity(pick(all_of(pigment_cols))),
    Pielou_pig = Shannon_pigment / log(vegan::specnumber(pick(all_of(pigment_cols)))),
    PigmentRichness = rowSums(pick(all_of(pigment_cols)) > 0)
  ) %>%
  select(time_month, Shannon_pigment, Pielou_pig, PigmentRichness)

# Zooplankton Data (wide format: columns suffixed _200 and _500)
zoo_ds <- readRDS("processed/Zoo_processed.rds") %>%
  mutate(time_month = format(date, format = "%m-%Y")) %>%
  select(-date)

# CTD Data
ctd_ds <- readRDS("processed/CTD_Isotherm21_MLD.rds")
ctd_ds$time_month <- format(ctd_ds$date, format = "%m-%Y")

# In 2012-11 there were two measurements, so average these:
ctd_ds2 <- ctd_ds %>%
  group_by(time_month) %>%
  summarize(Isotherm_21 = mean(Isotherm_21), MLD = mean(MLD), SST = mean(sst)) %>%
  ungroup()

# CTD Upwelling Index
ctd_up_ds <- readRDS("processed/CTD_UpwellingIndex.rds")
ctd_up_ds$time_month <- format(ctd_up_ds$date, format = "%m-%Y")

ctd_up_ds2 <- ctd_up_ds %>%
  group_by(time_month) %>%
  summarize(ui = first(ui), upwelling = first(upwelling)) %>%
  ungroup()

# Euphotic Depth
MLD2EuZ <- read.csv("processed/MLD2EuZ_2.csv")
MLD2EuZ$Date <- as.Date(MLD2EuZ$Date, format = "%Y-%m-%d")
MLD2EuZ$time_month <- format(MLD2EuZ$Date, format = "%m-%Y")
EuZ_ds <- MLD2EuZ %>% select(time_month, x1) %>% rename(euphotic_depth = x1)

# Sediment Trap Data
sedtrap_ds <- readRDS("processed/SedTrap_monthly_wide.rds")
# =============================================================================
# 2. CREATE COMPLETE MONTHLY BACKBONE
# =============================================================================

# Define CARIACO time series period
start_date <- as.Date("1995-11-01")
end_date <- as.Date("2017-01-01")

# Create complete monthly sequence
complete_months <- data.frame(
  date = seq(from = start_date, to = end_date, by = "month")
) %>%
  mutate(time_month = format(date, format = "%m-%Y"))

cat("Complete monthly backbone:", nrow(complete_months), "months\n")


# =============================================================================
# 3. JOIN ALL DATASETS TO MONTHLY BACKBONE
# =============================================================================

# Using the backbone as the base, left_join ensures we keep the exact monthly sequence
CARIACO_dat_joined <- list(
  complete_months,
  wind_ds,
  niskin_ds,
  hplc_ds,
  pigment_diversity,
  zoo_ds,
  ctd_ds2,
  ctd_up_ds2,
  EuZ_ds,
  amo_ds,
  meiv2_ds,
  sedtrap_ds
) %>%
  reduce(left_join, by = "time_month") %>%
  arrange(date)


# =============================================================================
# 4. TRUNCATE TO STUDY PERIOD
# =============================================================================

# Proper date filtering (matches Script 2 structure)
CARIACO_dat_joined_truncated <- CARIACO_dat_joined %>%
  filter(date >= start_date & date <= end_date) %>%
  select(-date)


# =============================================================================
# 5. DIAGNOSTICS
# =============================================================================

cat("\n=== Final dataset ===\n")
cat(sprintf("Rows: %d, Columns: %d\n", nrow(CARIACO_dat_joined_truncated), ncol(CARIACO_dat_joined_truncated)))

key_vars <- c("u10", "tp", "Chlorophyll", "PrimaryProductivity",
              "NO3_merged", "PO4_merged", "SiO4_merged",
              "Isotherm_21", "SST", "MLD",
              "micro", "nano", "pico", "size_centroid", "size_shannon",
              "Tot_Chl_a", "Shannon_pigment",
              "BIOMASS_200", "BIOMASS_500",
              "copepod_frac_200", "calanoid_frac_200")

na_summary <- CARIACO_dat_joined_truncated %>%
  select(any_of(key_vars)) %>%
  summarise(across(everything(), ~sum(is.na(.)))) %>%
  pivot_longer(everything(), names_to = "variable", values_to = "n_NA") %>%
  mutate(pct_NA = round(100 * n_NA / nrow(CARIACO_dat_joined_truncated), 1)) %>%
  arrange(desc(n_NA))

cat("\nMissing data summary (key variables):\n")
print(na_summary, n = 25)

cat("\n=== Column names ===\n")
cat(paste(names(CARIACO_dat_joined_truncated), collapse = "\n"))


# =============================================================================
# 6. SAVE
# =============================================================================

saveRDS(CARIACO_dat_joined_truncated, "processed/CARIACO_EnvData_combined.rds")
write.csv(CARIACO_dat_joined_truncated, "processed/CARIACO_EnvData_combined.csv")

cat("\n\nSaved to processed/CARIACO_EnvData_combined.rds and .csv\n")

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Lade nötiges Paket: permute



Complete monthly backbone: 255 months

=== Final dataset ===
Rows: 255, Columns: 110

Missing data summary (key variables):
# A tibble: 21 × 3
   variable             n_NA pct_NA
   <chr>               <int>  <dbl>
 1 calanoid_frac_200     157   61.6
 2 micro                 136   53.3
 3 nano                  136   53.3
 4 pico                  136   53.3
 5 size_centroid         136   53.3
 6 size_shannon          136   53.3
 7 BIOMASS_200           104   40.8
 8 BIOMASS_500           104   40.8
 9 copepod_frac_200      102   40  
10 Tot_Chl_a              93   36.5
11 Shannon_pigment        87   34.1
12 PrimaryProductivity    52   20.4
13 NO3_merged             45   17.6
14 PO4_merged             30   11.8
15 SiO4_merged            30   11.8
16 Chlorophyll            28   11  
17 Isotherm_21            27   10.6
18 SST                    27   10.6
19 MLD                    27   10.6
20 u10                     0    0  
21 tp                      0    0  

=== Column names ===
time_mo